# Kamera görüş açısının domates büyüme evresi sınıflandırmasına etkisi
**YOLO11n-cls ve YOLO26n-cls · TomatoMAP · 4 kamera · bitki gruplu 5 katlı çapraz doğrulama · geç füzyon**

**Kurulum**
1. `tomato_pipeline.py` dosyasını Drive'da `YOLODomatesEylul2026/code/` klasörüne yükleyin.
2. Çalışma zamanı: **GPU**, *Arka planda çalıştır* açık.
3. (Önerilir) 🔑 *Secrets* → `HF_TOKEN` (Hugging Face okuma token'ı).

**Oturum koparsa:** 1–3 numaralı hücreleri çalıştırın, sonra kaldığınız aşamaya dönün. Tamamlanan işler atlanır.

## 1. Drive ve paketler

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Sonuçların üretildiği sürümler sabitlenmiştir (yeniden üretilebilirlik için -U kullanılmaz)
!pip -q install ultralytics==8.4.155 "huggingface_hub>=1.32.0" scikit-learn==1.9.1 onnx onnxslim onnxruntime==1.30.0 openpyxl opencv-python-headless
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Modül

In [ ]:
import sys, importlib
sys.path.insert(0, '/content/drive/MyDrive/YOLODomatesEylul2026/code')
import tomato_pipeline as tp
importlib.reload(tp)

## 3. Yapılandırma
**Deneyler başladıktan sonra değiştirmeyin.** Kamera etiketlerine derece bilgisi, kaynaktan doğrulanmadan yazılmamalı.

In [ ]:
cfg = tp.Config(
    drive_root='/content/drive/MyDrive/YOLODomatesEylul2026',
    cameras=(1, 2, 3, 4),
    camera_labels={1: 'K1', 2: 'K2', 3: 'K3', 4: 'K4'},
    pose_seed=2026, fold_seed=2026, n_folds=5, train_seed=0,
    models=('yolo11n-cls.pt', 'yolo26n-cls.pt'),
    imgsz=320, epochs=50, patience=15, batch=64,
)
env = tp.setup(cfg)

## 4. AŞAMA 1 — İndeks ve veri denetimi
Beklenen: 64.464 görüntü, 101 bitki, 4 kamera, 12 poz. Oturum sayısı buradan okunur ve makalede bu değer kullanılır.

In [ ]:
full = tp.stage_index(cfg)

import json, os
audit_path = f"{cfg.drive_root}/results/tables/T0_metadata_audit.json"
with open(audit_path, encoding="utf-8") as f:
    audit = json.load(f)
print(json.dumps(audit, indent=2, ensure_ascii=False))

RESET_HINT = (f"\nDüzeltmeden sonra yeniden ayrıştırmak için şu dosyaları silin: "
              f"{cfg.drive_root}/cache/metadata_full.csv ve {audit_path}")

def require(cond, msg):
    if not cond:
        raise RuntimeError(msg + RESET_HINT)

require(audit["n_images"] == 64464, f"Beklenen 64.464 görüntü yerine {audit['n_images']} görüntü ayrıştırıldı.")
require(audit["n_plants"] == 101, f"Beklenen 101 bitki yerine {audit['n_plants']} bitki bulundu.")
require(audit["sessions_with_multiple_bbch"] == 0, f"{audit['sessions_with_multiple_bbch']} oturumda birden fazla BBCH etiketi bulundu.")
require(audit["cameras"] == [1, 2, 3, 4], f"Beklenmeyen kamera listesi: {audit['cameras']}")
require(audit["poses"] == list(range(1, 13)), f"Beklenmeyen poz listesi: {audit['poses']}")

print("✓ İndeks denetimi başarılı. Poz seçimi aşamasına geçilebilir.")

## 5. AŞAMA 2 — Oturum başına poz seçimi
Manifest: `manifests/selection_manifest.csv`

In [ ]:
sub = tp.stage_select(cfg)

import pandas as pd
man = pd.read_csv(f"{cfg.drive_root}/manifests/selection_manifest.csv")
excl_path = f"{cfg.drive_root}/manifests/excluded_sessions.csv"
n_excl = len(pd.read_csv(excl_path)) if os.path.exists(excl_path) else 0
print("Oturum:", len(man), "| Görüntü:", len(sub), "| Dışlanan oturum:", n_excl)
print(man["cls"].value_counts().sort_index())

if not man["session"].is_unique:
    raise RuntimeError("Seçim manifestinde yinelenen oturum var.")
if len(man) + n_excl != audit["n_sessions"]:
    raise RuntimeError(f"Seçilen ({len(man)}) + dışlanan ({n_excl}) oturum, denetimdeki {audit['n_sessions']} oturuma eşit değil.")
if len(sub) != len(cfg.cameras) * len(man):
    raise RuntimeError(f"Alt küme {len(sub)} görüntü; beklenen {len(cfg.cameras) * len(man)}.")
if sorted(man["cls"].unique()) != sorted(tp.CLASSES):
    raise RuntimeError(f"Alt kümede eksik sınıf var: {sorted(set(tp.CLASSES) - set(man['cls']))}")
if n_excl:
    print(f"⚠ {n_excl} oturum dört kamerada ortak poz olmadığı için dışlandı; makalede raporlanmalı.")
print("✓ Poz seçimi denetimi başarılı.")

## 6. AŞAMA 3 — Görüntüler
Önce yalnızca seçilen ~5.400 dosya indirilir; olmazsa tam indirmeye geçer.

In [ ]:
tp.stage_images(cfg)

## 7. AŞAMA 4 — Bitki düzeyinde 5 kat
Her katta 6 sınıfın bulunduğu denetlenir.

In [ ]:
tp.stage_folds(cfg)

## 8. Ağırlık kontrolü

In [ ]:
from ultralytics import YOLO
for m in cfg.models:
    YOLO(m).info()

## 9. AŞAMA 5 — 40 eğitim
Kesilirse hücreyi tekrar çalıştırın.

In [ ]:
tp.stage_train_eval(cfg)
tp.status(cfg)

## 10. AŞAMA 6 — OOF ve geç füzyon

In [ ]:
tp.stage_oof(cfg)

## 11. AŞAMA 7 — Maliyet (tek kamera vs dört kamera)

In [ ]:
tp.stage_efficiency(cfg)

## 12. AŞAMA 8 — Tablolar, şekiller, istatistik

In [ ]:
tp.stage_analysis(cfg)
import pandas as pd
pd.read_csv(f'{cfg.drive_root}/results/tables/T3b_main_results_formatted.csv')

## 13. Paketle
`results_package.zip` dosyasını Claude'a iletin.

In [ ]:
import shutil, os
root, pkg = cfg.drive_root, '/content/results_package'
shutil.rmtree(pkg, ignore_errors=True)
for sub_dir in ('results', 'logs', 'manifests'):
    shutil.copytree(f'{root}/{sub_dir}', f'{pkg}/{sub_dir}')
os.makedirs(f'{pkg}/training_curves', exist_ok=True)
for run in os.listdir(f'{root}/runs'):
    f = f'{root}/runs/{run}/results.csv'
    if os.path.exists(f):
        shutil.copy(f, f'{pkg}/training_curves/{run}.csv')
shutil.make_archive(f'{root}/results_package', 'zip', pkg)
print('Hazır:', f'{root}/results_package.zip')